# SEAL

Replicates **"SEAL: Steerable Reasoning Calibration of Large Language Models for Free"** ([arXiv:2504.07986](https://arxiv.org/abs/2504.07986)) on DeepSeek-R1-Distill-Qwen-1.5B, end to end in one engine:

1. **Construction** — chain-of-thought traces are generated, each paragraph segment is classified as execution / reflection / transition by keyword, hidden states are captured at the paragraph-break tokens, and each category is averaged into a control vector (`execution_avg_vector.gguf` / `reflection_avg_vector.gguf` / `transition_avg_vector.gguf`).
2. **Steering** — promoting execution thoughts while suppressing reflection and transition thoughts, only at paragraph-break tokens during generation, trims redundant chain-of-thought, reported as the mean generated length over 100 MATH-500 problems (`math500.json`).

Uses the EasySteer v2 steering API (`SteeringSpec` / `VectorSpec` / `ApplySpec`).

In [1]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

from vllm import LLM, SamplingParams
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = "/home/shenyl/hf/model/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/"  # deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B

# One engine serves both construction (capture) and steering. The
# three-vector SEAL spec is a multi-vector workload — declare it and
# the engine derives the graph integration that can serve it.
llm = LLM(
    model=MODEL,
    enable_steer_vector=True,
    steer_algorithms=["direct"],
    steer_multi_vector=True,
)
tokenizer = llm.get_tokenizer()

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


(EngineCore pid=3929934) 

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=3929934) 

...

Capturing CUDA graphs (PIECEWISE):  82%|████████▏ | 42/51 [00:02<00:00, 19.99it/s]

Capturing CUDA graphs (PIECEWISE):  88%|████████▊ | 45/51 [00:02<00:00, 19.93it/s]

Capturing CUDA graphs (PIECEWISE):  94%|█████████▍| 48/51 [00:02<00:00, 18.72it/s]

Capturing CUDA graphs (PIECEWISE):  98%|█████████▊| 50/51 [00:02<00:00, 18.96it/s]

Capturing CUDA graphs (PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 19.47it/s]

## Vector construction

### Generate reasoning traces

In [2]:
problems = [
    "Chandra has four bowls.  Each one is a different color (red, blue, yellow, green).  She also has exactly one glass the same color as each bowl.  If she chooses a bowl and a glass from the cupboard, how many pairings are possible?  One such pairing is a blue bowl and a yellow glass.",
    "The distance between two cities on a map is 15 inches. If the scale is 0.25 inches = 3 miles, how many miles apart are the actual cities?",
    "How many prime numbers are between 20 and 30?",
    "A rectangle has a perimeter of 30 units and its width is 6 units. What is its area?",
    "If 3x + 7 = 25, what is the value of 2x - 1?",
    "A bag contains 4 red marbles and 6 blue marbles. What is the probability of drawing a red marble?",
    "What is the least common multiple of 12 and 18?",
    "A train travels 240 miles in 4 hours. At the same speed, how far does it travel in 7 hours?",
    "The sum of three consecutive integers is 48. What is the largest of the three?",
    "What is the value of 2^5 + 3^3?",
]
texts = ["Please reason step by step, and put your final answer within \\boxed{}.\nUser: " + p + "\nAssistant: <think>" for p in problems]

answers = llm.generate(
    texts,
    SamplingParams(temperature=0, max_tokens=4096, skip_special_tokens=False),
    use_tqdm=False,
)
qa_pairs = [t + a.outputs[0].text for t, a in zip(texts, answers)]

### Classify the paragraph breaks

SEAL categorizes each `\n\n`-delimited reasoning segment by keyword: **transition** (switching approach), **reflection** (checking work), everything else **execution**. Each segment's category is attributed to the paragraph-break token that opens it.

In [3]:
TRANSITION_KEYWORDS = [
    "alternatively", "think differently", "another way", "another approach",
    "another method", "another solution", "another strategy", "another technique",
]
REFLECTION_KEYWORDS = [
    "wait", "verify", "make sure", "hold on", "think again", "'s correct",
    "'s incorrect", "let me check", "seems right",
]


def classify(segment):
    lower = segment.lower()
    if any(k in lower for k in TRANSITION_KEYWORDS):
        return "Transition"
    if any(k in lower for k in REFLECTION_KEYWORDS):
        return "Reflection"
    return "Execution"


# "\n\n" tokenizes to tokens ending in the "ĊĊ" suffix; those are the
# paragraph-break positions whose hidden states SEAL uses.
all_ids = []
category_by_position = []
for qa in qa_pairs:
    ids = tokenizer(qa, add_special_tokens=True).input_ids
    tokens = tokenizer.convert_ids_to_tokens(ids)
    positions = [i for i, t in enumerate(tokens) if t.endswith("ĊĊ")]
    by_pos = {}
    for j, pos in enumerate(positions):
        end = positions[j + 1] if j + 1 < len(positions) else len(ids)
        segment = tokenizer.decode(ids[pos + 1:end], skip_special_tokens=True)
        by_pos[pos] = classify(segment.strip())
    all_ids.append(ids)
    category_by_position.append(by_pos)
    counts = {c: sum(v == c for v in by_pos.values())
              for c in ("Execution", "Reflection", "Transition")}
    print(counts)

{'Execution': 31, 'Reflection': 39, 'Transition': 4}
{'Execution': 14, 'Reflection': 1, 'Transition': 1}
{'Execution': 16, 'Reflection': 3, 'Transition': 0}
{'Execution': 35, 'Reflection': 2, 'Transition': 0}
{'Execution': 61, 'Reflection': 1, 'Transition': 40}
{'Execution': 13, 'Reflection': 4, 'Transition': 7}
{'Execution': 2, 'Reflection': 0, 'Transition': 0}
{'Execution': 17, 'Reflection': 1, 'Transition': 2}
{'Execution': 39, 'Reflection': 1, 'Transition': 2}
{'Execution': 11, 'Reflection': 2, 'Transition': 0}


### Capture and average

The `tokens` filter selects only rows whose input token ends in `ĊĊ`, so the engine ships just the paragraph-break hidden states. Each category's rows are averaged per layer into one control vector.

In [4]:
import easysteer.hidden_states as hs
from vllm.steer_vectors.api import SelectSpec

newline_ids = sorted(
    tid for tok_str, tid in tokenizer.get_vocab().items()
    if tok_str.endswith("ĊĊ")
)

result = hs.capture(
    llm,
    [{"prompt_token_ids": ids} for ids in all_ids],
    select=SelectSpec(prompt_tokens=newline_ids),
)

In [5]:
import numpy as np

from easysteer.steer import StatisticalControlVector

collected = {"Transition": {}, "Reflection": {}, "Execution": {}}
for i in range(len(result)):
    rows = result.sample(i)
    for row_idx, pos in enumerate(result.sample_positions(i)):
        # Every captured row must map to a classified paragraph break;
        # a miss means client/engine tokenization desynced.
        category = category_by_position[i][pos]
        for layer_id, tensor in rows.items():
            collected[category].setdefault(layer_id, []).append(
                tensor[row_idx].float().numpy()
            )

for category, per_layer in collected.items():
    directions = {
        layer_id: np.mean(np.stack(states), axis=0)
        for layer_id, states in per_layer.items()
    }
    n = len(next(iter(per_layer.values())))
    control_vector = StatisticalControlVector(
        method="Average",
        directions=directions,
        metadata={"num_vectors_averaged": n},
    )
    control_vector.export_gguf(f"{category.lower()}_avg_vector.gguf")
    print(f"{category}: averaged {n} rows")

Transition: averaged 56 rows


Reflection: averaged 54 rows


Execution: averaged 239 rows


## Steering

In [6]:
# Baseline: no steering. As in the experiment section, evaluate on
# 100 MATH-500 problems and report only the mean generated length —
# individual greedy trajectories vary between engine boots, the
# aggregate does not.
import json

with open("math500.json", encoding="utf-8") as f:
    eval_problems = [x["problem"] for x in json.load(f)][:100]
eval_texts = [
    "Please reason step by step, and put your final answer within "
    "\\boxed{}.\nUser: " + p + "\nAssistant: <think>"
    for p in eval_problems
]
params = SamplingParams(temperature=0, max_tokens=8192, skip_special_tokens=False)


def mean_tokens(outputs):
    return sum(len(o.outputs[0].token_ids) for o in outputs) / len(outputs)


baseline = mean_tokens(llm.generate(eval_texts, params, use_tqdm=False))
print(f"Baseline mean tokens: {baseline:.0f}")

Baseline mean tokens: 4397


In [7]:
# Promote execution thoughts (+), suppress reflection and transition (-),
# all three applied at layer 20 and only at the paragraph-break ("\n\n")
# tokens during generation — the same newline-suffixed ids the capture
# selected. conflict="sequential" stacks the three vectors. Scale 0.5:
# the effect is non-monotone, and 0.75+ can tip greedy decoding into
# runaway reasoning instead of trimming it.
steering = SteeringSpec(
    conflict="sequential",
    vectors=[
        VectorSpec(
            source="execution_avg_vector.gguf",
            scale=0.5,
            layers=[20],
            apply=ApplySpec(generation_tokens=newline_ids),
        ),
        VectorSpec(
            source="reflection_avg_vector.gguf",
            scale=-0.5,
            layers=[20],
            apply=ApplySpec(generation_tokens=newline_ids),
        ),
        VectorSpec(
            source="transition_avg_vector.gguf",
            scale=-0.5,
            layers=[20],
            apply=ApplySpec(generation_tokens=newline_ids),
        ),
    ],
)

steered = mean_tokens(llm.generate(eval_texts, params, steering=steering,
                                   use_tqdm=False))
print(f"SEAL mean tokens: {steered:.0f} ({(steered / baseline - 1) * 100:+.0f}%)")

SEAL mean tokens: 3287 (-25%)
